# Лабораторная работа 2. Сэмплинг

## Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.over_sampling import RandomOverSampler, SMOTE

sns.set_style("whitegrid")

## Загрузка данных

In [ ]:
dataset_path = "../../datasets/gtzan/features_30_sec.csv"
data = pd.read_csv(dataset_path)

print(f"Размер датасета: {data.shape[0]} строк, {data.shape[1]} столбцов")
data.head()

## Описательная статистика

In [ ]:
data.describe()

## Анализ баланса классов

Датасет GTZAN содержит 10 жанров по 100 записей каждый — идеально сбалансированный набор. Для демонстрации методов сэмплинга **искусственно дебалансируем каждый из 10 классов**, оставив случайную долю записей. Так получится правдоподобный сценарий многоклассовой задачи с разными размерами классов — на нём `imblearn` (RUS, ROS, SMOTE, Tomek Links) показывают свою работу куда более реалистично, чем на бинарной jazz-vs-остальные.

In [ ]:
# 1. Случайные пропорции для каждого жанра (фиксированный seed → воспроизводимо)
GENRES = sorted(data["label"].unique())
rng = np.random.default_rng(42)
imbalance_ratios = {g: rng.uniform(0.1, 1.0) for g in GENRES}
keep_counts = {g: max(10, int(round(100 * r))) for g, r in imbalance_ratios.items()}

# 2. Создаём дебалансированный датасет: для каждого жанра берём свою долю записей
imbalanced = pd.concat([
    data[data["label"] == g].sample(n=keep_counts[g], random_state=42)
    for g in GENRES
]).reset_index(drop=True)

# 3. Визуализация: сравниваем исходное и новое распределение
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

original_counts = data["label"].value_counts().reindex(GENRES)
sns.barplot(x=original_counts.index, y=original_counts.values, palette="viridis", ax=axes[0])
axes[0].set_title("Исходное распределение (сбалансированный GTZAN)")
axes[0].set_xlabel("Жанр"); axes[0].set_ylabel("Количество")
axes[0].tick_params(axis="x", rotation=45)
axes[0].axhline(100, color="gray", linestyle=":", alpha=0.6)

imb_counts = imbalanced["label"].value_counts().reindex(GENRES)
sns.barplot(x=imb_counts.index, y=imb_counts.values, palette="tab10", ax=axes[1])
axes[1].set_title("После искусственного мульти-классового дисбаланса")
axes[1].set_xlabel("Жанр"); axes[1].set_ylabel("Количество")
axes[1].tick_params(axis="x", rotation=45)
axes[1].axhline(100, color="red", linestyle="--", alpha=0.6, label="исходный размер (100)")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Размеры классов после дебаланса:")
for g in GENRES:
    print(f"  {g:10s}: {imb_counts[g]:3d}  (доля {imbalance_ratios[g]:.2f})")
print(f"\nВсего записей: {len(imbalanced)} (было 1000)")
print(f"Min: {imb_counts.min()} ({imb_counts.idxmin()}), "
      f"Max: {imb_counts.max()} ({imb_counts.idxmax()})")
print(f"Imbalance ratio (max/min): {imb_counts.max()/imb_counts.min():.2f}")

## Правило NEPV (Number of Events Per Variable)

Правило NEPV определяет минимальный размер выборки для надёжного обучения модели. Рекомендуемое значение: **NEPV ≥ 10**, то есть на каждую независимую переменную должно приходиться не менее 10 наблюдений миноритарного класса.

Для нашей задачи (мульти-класс):
- Количество признаков: **57** (исключаем `filename` и `label`)
- Размер самого редкого класса: `min(keep_counts.values())`
- **NEPV = min_class_size / 57**

Если NEPV < 10 — объём миноритарного класса недостаточен для надёжной модели на всех 57 признаках. Это мотивирует применение методов оверсэмплинга (SMOTE, ROS) для увеличения количества наблюдений.

## Подготовка признаков

In [ ]:
feature_cols = imbalanced.columns.drop(["filename", "label"])
X = imbalanced[feature_cols]
y = imbalanced["label"]

print(f"Признаки: {X.shape[1]} столбцов")
print(f"Выборка: {X.shape[0]} строк (10 классов с разными размерами)")
print(f"\nNEPV = {y.value_counts().min()} / {X.shape[1]} = "
      f"{y.value_counts().min() / X.shape[1]:.2f}  "
      f"(порог 10 — {'достаточно' if y.value_counts().min() / X.shape[1] >= 10 else 'НЕдостаточно'})")

## Простой случайный сэмплинг

In [ ]:
sample = imbalanced.sample(n=100, random_state=42)

print(f"Размер выборки: {len(sample)}")
print(f"\nРаспределение жанров в случайной выборке (без стратификации):")
print(sample["label"].value_counts())
print(f"\nИз-за дисбаланса исходной выборки случайная подвыборка")
print(f"наследует те же пропорции — крупные классы доминируют.")

## Стратифицированное разбиение на обучающую и тестовую выборки

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

print(f"Обучающая выборка: {X_train.shape[0]} строк")
print(f"Тестовая выборка:  {X_test.shape[0]} строк")
print(f"\nРаспределение классов в обучающей выборке:")
print(y_train.value_counts().sort_index())
print(f"\nРаспределение классов в тестовой выборке:")
print(y_test.value_counts().sort_index())
print(f"\nМин. размер класса в train: {y_train.value_counts().min()}")
print(f"Мин. размер класса в test:  {y_test.value_counts().min()}")

## Методы ресэмплинга

Далее применим методы балансировки классов к **обучающей выборке** (ресэмплинг применяется только к train, не к test):

1. **RandomUnderSampler** — случайный андерсэмплинг
2. **RandomOverSampler** — случайный оверсэмплинг
3. **SMOTE** — синтетический оверсэмплинг
4. **Tomek Links** — андерсэмплинг на основе удаления пограничных пар

### 1. Случайный андерсэмплинг (RandomUnderSampler)

In [ ]:
rus = RandomUnderSampler(sampling_strategy="not minority", random_state=42)
X_under, y_under = rus.fit_resample(X_train, y_train)

print(f"До андерсэмплинга:    {len(y_train)} строк")
print(f"После андерсэмплинга: {len(y_under)} строк")
print(f"Удалено строк: {len(y_train) - len(y_under)}")
print(f"\nРаспределение классов после RUS:")
print(y_under.value_counts().sort_index())
print(f"\n«not minority» => все классы кроме самого маленького "
      f"уменьшаются до его размера ({y_train.value_counts().min()}).")

### 2. Случайный оверсэмплинг (RandomOverSampler)

In [ ]:
ros = RandomOverSampler(sampling_strategy="not majority", random_state=42)
X_over, y_over = ros.fit_resample(X_train, y_train)

print(f"До оверсэмплинга:    {len(y_train)} строк")
print(f"После оверсэмплинга: {len(y_over)} строк")
print(f"Добавлено строк: {len(y_over) - len(y_train)}")
print(f"\nРаспределение классов после ROS:")
print(y_over.value_counts().sort_index())
print(f"\n«not majority» => все классы кроме самого большого "
      f"дублируются до его размера ({y_train.value_counts().max()}).")

### 3. SMOTE (Synthetic Minority Over-sampling Technique)

In [ ]:
k_neighbors = min(5, y_train.value_counts().min() - 1)
smote = SMOTE(
    sampling_strategy="not majority",
    k_neighbors=k_neighbors,
    random_state=42,
)
X_smote, y_smote = smote.fit_resample(X_train, y_train)

print(f"До SMOTE:    {len(y_train)} строк")
print(f"После SMOTE: {len(y_smote)} строк")
print(f"Синтезировано строк: {len(y_smote) - len(y_train)}")
print(f"\nРаспределение классов после SMOTE:")
print(y_smote.value_counts().sort_index())
print(f"\nИспользован k_neighbors={k_neighbors} "
      f"(подобран по min классу train = {y_train.value_counts().min()}).")

### 4. Tomek Links

In [ ]:
tl = TomekLinks(sampling_strategy="auto")
X_tomek, y_tomek = tl.fit_resample(X_train, y_train)

print(f"До Tomek Links:    {len(y_train)} строк")
print(f"После Tomek Links: {len(y_tomek)} строк")
print(f"Удалено строк: {len(y_train) - len(y_tomek)}")
print(f"\nРаспределение классов после Tomek Links:")
print(y_tomek.value_counts().sort_index())
print(f"\n«auto» => удаляются Tomek-связи в любом классе. Метод не "
      f"балансирует классы полностью, а очищает границы между ними.")

## Сравнение результатов

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 5), sharey=True)

datasets = [
    ("Исходная (train)", y_train),
    ("RandomUnderSampler", y_under),
    ("RandomOverSampler", y_over),
    ("SMOTE", y_smote),
    ("Tomek Links", y_tomek),
]

palette = sns.color_palette("tab10", n_colors=len(GENRES))
genre_to_color = dict(zip(GENRES, palette))

for ax, (title, target) in zip(axes, datasets):
    counts = pd.Series(target).value_counts().reindex(GENRES, fill_value=0)
    bars = ax.bar(counts.index, counts.values, color=[genre_to_color[g] for g in counts.index])
    ax.set_title(f"{title}\n(n={len(target)})", fontsize=11)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    for bar in bars:
        ax.annotate(
            f"{int(bar.get_height())}",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            ha="center", va="bottom", fontsize=7,
        )

axes[0].set_ylabel("Количество треков")
plt.suptitle("Распределение 10 жанров после различных методов ресэмплинга", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
def stats(name, target):
    counts = pd.Series(target).value_counts()
    return {
        "Метод": name,
        "Всего": len(target),
        "Min класс": counts.min(),
        "Max класс": counts.max(),
        "Std": counts.std().round(1),
        "Imbalance ratio (max/min)": round(counts.max() / counts.min(), 2),
    }

results = pd.DataFrame([
    stats("Исходная (train)",   y_train),
    stats("RandomUnderSampler", y_under),
    stats("RandomOverSampler",  y_over),
    stats("SMOTE",              y_smote),
    stats("Tomek Links",        y_tomek),
])
results

## Анализ распределения признаков после SMOTE

In [ ]:
min_class = y_train.value_counts().idxmin()
compare_features = ["tempo", "spectral_centroid_mean", "mfcc1_mean", "rms_mean"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, feat in zip(axes.ravel(), compare_features):
    df_orig = pd.DataFrame({
        "value": X_train.loc[y_train == min_class, feat].values,
        "source": "Исходные",
    })
    df_sm = pd.DataFrame({
        "value": X_smote.loc[y_smote == min_class, feat].values,
        "source": "После SMOTE",
    })
    df_plot = pd.concat([df_orig, df_sm])

    sns.violinplot(data=df_plot, x="source", y="value",
                   palette=["#e74c3c", "#f39c12"], ax=ax)
    ax.set_title(f"{feat} (класс «{min_class}»)", fontsize=11)
    ax.set_xlabel(""); ax.set_ylabel(feat)

plt.suptitle(
    f"Сравнение распределений до и после SMOTE\n"
    f"(самый редкий класс «{min_class}», "
    f"{(y_train == min_class).sum()} → {(y_smote == min_class).sum()} наблюдений)",
    fontsize=13, y=1.02,
)
plt.tight_layout()
plt.show()

## Проверка SMOTE на дубликаты

In [ ]:
def per_class_dups(X_resampled, y_resampled, y_orig):
    rows = []
    for cls in sorted(y_resampled.unique()):
        mask_res = (y_resampled == cls)
        n_total = int(mask_res.sum())
        n_orig = int((y_orig == cls).sum())
        n_synthetic = n_total - n_orig
        n_dups = int(X_resampled.loc[mask_res].duplicated().sum())
        rows.append({
            "class": cls,
            "n_orig (train)": n_orig,
            "n_total (resampled)": n_total,
            "n_synthetic": max(n_synthetic, 0),
            "n_duplicates": n_dups,
        })
    return pd.DataFrame(rows)

print("=== SMOTE ===  (синтезирует промежуточные точки, не дубликаты)")
print(per_class_dups(X_smote, y_smote, y_train).to_string(index=False))

print("\n=== RandomOverSampler ===  (тупо копирует наблюдения)")
print(per_class_dups(X_over, y_over, y_train).to_string(index=False))

## Сравнение средних значений признаков

In [ ]:
# Сравним средние самого редкого класса до и после ресэмплинга
min_class = y_train.value_counts().idxmin()
compare_cols = ["tempo", "spectral_centroid_mean", "rms_mean", "mfcc1_mean", "mfcc2_mean"]

means = pd.DataFrame({
    "Исходные (train)": X_train.loc[y_train == min_class, compare_cols].mean(),
    "UnderSampler":     X_under.loc[y_under == min_class, compare_cols].mean(),
    "OverSampler":      X_over.loc[y_over == min_class, compare_cols].mean(),
    "SMOTE":            X_smote.loc[y_smote == min_class, compare_cols].mean(),
    "Tomek Links":      X_tomek.loc[y_tomek == min_class, compare_cols].mean(),
})

print(f"Средние значения признаков для самого редкого класса «{min_class}»:")
means.round(3)

## Выводы

1. **Многоклассовый случайный дисбаланс.** Исходный GTZAN сбалансирован (10 жанров × 100). Для демонстрации сэмплинга мы искусственно дебалансировали каждый из 10 классов: для каждого жанра случайно (с фиксированным `random_state`) выбрана своя доля записей в диапазоне 10–100. Так получился реалистичный мультиклассовый сценарий с разным размером классов вместо тривиального «бинарного» случая.

2. **Правило NEPV.** При 57 признаках и нескольких десятках наблюдений в самых редких классах NEPV оказывается заметно ниже порога 10 — это формальный сигнал, что миноритарным классам не хватает данных для надёжной модели.

3. **Стратифицированное разбиение** сохранило пропорции 10 классов в `train` и `test` — без него редкие классы могли бы вообще исчезнуть из теста.

4. **Поведение методов на 10 классах:**
   - **RandomUnderSampler (`not minority`)**: уравнял всё до размера самого редкого класса — общий объём схлопывается, теряется много данных из крупных классов.
   - **RandomOverSampler (`not majority`)**: дублирует наблюдения редких классов до размера самого крупного. Балансирует, но создаёт массу одинаковых строк.
   - **SMOTE (`not majority`)**: то же выравнивание по верхней границе, но синтезирует промежуточные точки между ближайшими соседями (`k_neighbors` подобран по min классу), без точных дубликатов.
   - **Tomek Links (`auto`)**: чистит границы между классами, удаляя пары «соседей разных классов». Не балансирует, но улучшает разделимость — полезный preprocessing перед обучением.

5. **Что критически важно при мульти-классе:**
   - подобрать `k_neighbors` SMOTE по самому редкому классу (иначе падает с ошибкой);
   - стратифицировать `train_test_split` по `y` — иначе редкие классы могут исчезнуть из теста;
   - смотреть на размеры **всех** классов (Min/Max/Std/Imbalance ratio), а не только на «класс 0 / класс 1».